Route 53 is AWS's managed DNS service — it translates domain names into IP addresses, routes traffic intelligently across regions and endpoints, and monitors the health of your resources. It is one of the most heavily tested networking services on the SAA-C03 exam, particularly the routing policies and health check integration.

## DNS Fundamentals

Before diving into Route 53, a quick refresher on how DNS works:

```text
Browser: "What is the IP of api.example.com?"
  │
  ▼
Recursive Resolver (your ISP or 8.8.8.8)
  │  asks Root DNS → .com TLD → example.com Name Servers
  ▼
Route 53 Name Server: "It's 1.2.3.4" (with TTL)
  │
  ▼
Browser caches result for TTL seconds, connects to 1.2.3.4
```

**TTL (Time to Live)** — how long resolvers cache the answer. Lower TTL = faster DNS changes propagate but more queries hit Route 53 (more cost). Higher TTL = fewer queries but slower failover.

### DNS record types

| Record | Purpose | Example |
|---|---|---|
| **A** | Maps hostname → IPv4 | `api.example.com → 1.2.3.4` |
| **AAAA** | Maps hostname → IPv6 | `api.example.com → 2001:db8::1` |
| **CNAME** | Maps hostname → another hostname | `www.example.com → example.com` |
| **NS** | Name servers for the hosted zone | Delegated by registrar |
| **MX** | Mail exchange servers | Email routing |
| **TXT** | Text data | Domain verification, SPF records |
| **SRV** | Service location | Port + priority for services |
| **CAA** | Certificate authority authorisation | Which CAs can issue certs |

> **CNAME limitation**: A CNAME cannot point to a **zone apex** (the root domain itself — `example.com`). You can't create a CNAME for `example.com`, only for subdomains like `www.example.com`. This is where **Alias records** come in.

## Hosted Zones

A **hosted zone** is a container for DNS records for a domain. It tells Route 53 how to respond to queries for that domain and its subdomains.

### Public hosted zone
- Answers DNS queries from the **public internet**
- Used when your domain is publicly accessible
- Cost: $0.50/month per hosted zone

### Private hosted zone
- Answers DNS queries **only from within one or more VPCs**
- Used for internal service discovery — e.g. `api.internal` resolves to a private IP
- VPC must have `enableDnsSupport` and `enableDnsHostnames` enabled
- Can be associated with VPCs across accounts
- Cost: $0.50/month per hosted zone

```text
Public hosted zone:   example.com        → internet-facing
Private hosted zone:  internal.corp      → VPC-only, resolves to private IPs
```

## Alias Records

An **Alias record** is a Route 53 extension that maps a hostname to an **AWS resource** (not a raw IP). It solves the CNAME-at-zone-apex problem.

### Alias vs CNAME

| | CNAME | Alias |
|---|---|---|
| Points to | Any hostname | AWS resource only |
| Zone apex (`example.com`) | ❌ Not allowed | ✅ Allowed |
| DNS query charges | Yes | **Free** |
| TTL | You set it | AWS manages it |
| Health check integration | No | Yes |

### Alias targets (AWS resources)
- Elastic Load Balancers (ALB, NLB, CLB)
- CloudFront distributions
- API Gateway
- S3 static website endpoints
- Elastic Beanstalk environments
- VPC interface endpoints
- **Another Route 53 record in the same hosted zone**

> **Exam tip:** You cannot create an Alias record pointing to an EC2 DNS name or an RDS endpoint — those require a CNAME.

## Routing Policies

Routing policies control how Route 53 responds to DNS queries. This is the most exam-heavy section of Route 53.

### Simple
Returns a single resource. If multiple values are specified, the client receives all of them and picks one randomly (client-side load balancing).
- **No health checks**
- Use case: single resource, no failover needed

### Weighted
Splits traffic across multiple resources by weight. Weights are relative — weight 70 + weight 30 = 70% / 30% split.
- Set weight to **0** to stop sending traffic to a resource
- If all weights are 0, traffic is distributed equally
- Supports health checks — unhealthy targets are excluded
- Use case: **A/B testing**, blue/green deployments, canary releases, gradual migration

```text
api.example.com
  ├── 10.0.1.1  weight=70  → 70% of traffic
  └── 10.0.2.1  weight=30  → 30% of traffic
```

### Latency-based
Routes to the AWS region with the **lowest latency** for the user — based on AWS's latency measurements between regions and end-user locations.
- Does NOT use geographic location — a user in Europe might be routed to us-east-1 if latency is lower
- Supports health checks
- Use case: **multi-region active-active**, lowest response time for global users

### Failover
Active/passive setup with a **primary** and **secondary** record.
- Route 53 health check monitors the primary
- If primary is unhealthy → all traffic automatically routes to secondary
- Use case: **disaster recovery**, hot standby

```text
api.example.com
  ├── PRIMARY   → us-east-1 ALB  (health checked)
  └── SECONDARY → eu-west-1 ALB  (standby — receives traffic only if primary fails)
```

### Geolocation
Routes based on the **geographic location of the user** (continent, country, or US state).
- Unlike latency-based, this is strictly about location — not performance
- Create a **default record** for users who don't match any location rule
- Use case: **data sovereignty** (keep EU traffic in EU), localised content, blocking regions

### Geoproximity
Routes based on geographic location of users **and resources**, with a **bias** to shift more or less traffic to a resource.
- Bias +1 to +99: expand the region's coverage area (attract more traffic)
- Bias -1 to -99: shrink the coverage area
- Requires **Route 53 Traffic Flow** (visual policy editor)
- Use case: shift traffic gradually from one region to another

### IP-based
Routes based on the **client's IP address** (CIDR ranges you define).
- You provide a list of CIDR → endpoint mappings
- Use case: route corporate office traffic (known IP range) to an internal endpoint; ISP-based optimisation

### Multi-value
Returns up to **8 healthy records** chosen randomly. Similar to Simple with multiple values, but:
- Supports **health checks** — only healthy records are returned
- Not a replacement for a load balancer, but provides client-side load balancing with health awareness
- Use case: simple load distribution across a small number of servers

### Routing policy comparison

| Policy | Decision basis | Health checks | Use case |
|---|---|---|---|
| Simple | None (single or random) | ❌ | Single resource |
| Weighted | Assigned weights | ✅ | A/B testing, canary |
| Latency | Lowest latency to region | ✅ | Multi-region performance |
| Failover | Primary/secondary | ✅ Required | DR, active-passive |
| Geolocation | User's country/continent | ✅ | Compliance, localisation |
| Geoproximity | Distance + bias | ✅ | Traffic shifting between regions |
| IP-based | Client CIDR | ✅ | Known IP ranges |
| Multi-value | Random (healthy only) | ✅ | Simple HA without ELB |

## Health Checks

Route 53 health checkers are distributed globally — they probe your endpoint from multiple locations and determine if it's healthy based on a threshold (default: 3 of 18+ checkers must agree it's unhealthy).

### Types of health checks

| Type | What it checks |
|---|---|
| **Endpoint** | HTTP/HTTPS/TCP to a specific IP or hostname. Passes if returns 2xx/3xx within 4 seconds. |
| **Calculated** | Combines multiple health checks with AND/OR/NOT logic. Useful for checking dependent services. |
| **CloudWatch alarm** | Monitors a CloudWatch metric alarm state. Used for resources that can't be probed directly (e.g. private resources, DynamoDB). |

### Health checks for private resources
Route 53 health checkers are on the **public internet** — they cannot reach private VPC resources directly. To monitor a private endpoint:
1. Create a **CloudWatch metric** for the resource (e.g. ALB healthy host count)
2. Create a **CloudWatch alarm** on that metric
3. Create a Route 53 **CloudWatch alarm health check** monitoring that alarm

### Health check settings
- **Interval**: 30 seconds (standard) or 10 seconds (fast, higher cost)
- **Threshold**: number of consecutive failures before marking unhealthy (default: 3)
- **String matching**: check if response body contains a specific string (first 5120 bytes)

## Domain Registration & Transfer

Route 53 is also a **domain registrar** — you can register domains directly through it.

- Register new domains (`.com`, `.io`, `.org`, etc.) — starts at ~$13/year
- Transfer existing domains from other registrars
- When you register/transfer a domain, Route 53 automatically creates a **public hosted zone** and sets the NS records at the registrar

### Using Route 53 DNS with a third-party registrar
If your domain is registered elsewhere (GoDaddy, Namecheap), you can still use Route 53 as the DNS provider:
1. Create a public hosted zone in Route 53
2. Note the 4 NS records Route 53 assigns
3. Update the name servers at your registrar to those 4 NS records

### DNSSEC
Route 53 supports **DNSSEC signing** for public hosted zones — adds cryptographic signatures to DNS responses to prevent DNS spoofing/cache poisoning.

## Route 53 Resolver

Every VPC has a built-in DNS resolver at the VPC base CIDR + 2 (e.g. `10.0.0.2`). This is the **Route 53 Resolver**.

For hybrid environments (VPC + on-premises), you need DNS queries to flow in both directions:

### Resolver Endpoints

| Endpoint | Direction | Use case |
|---|---|---|
| **Inbound** | On-premises → AWS | On-premises servers resolve AWS private hosted zone records |
| **Outbound** | AWS → On-premises | EC2 instances resolve on-premises DNS hostnames |

```text
On-premises DNS server
  │  "What is db.internal.corp?"
  ▼
Route 53 Inbound Endpoint (ENI in VPC)
  │  forwards to Route 53 Resolver
  ▼
Private Hosted Zone: db.internal.corp → 10.0.3.45

EC2 Instance
  │  "What is printer.office.corp?"
  ▼
Route 53 Outbound Endpoint
  │  forwarding rule: *.office.corp → on-premises DNS 192.168.1.53
  ▼
On-premises DNS: printer.office.corp → 192.168.50.12
```

Resolver Rules define which domains are forwarded to which DNS servers.

## Working with Route 53 via boto3

In [ ]:
import boto3
import uuid

r53 = boto3.client('route53')

In [ ]:
# Create a public hosted zone
response = r53.create_hosted_zone(
    Name='example.com',
    CallerReference=str(uuid.uuid4()),  # unique string to prevent duplicates
    HostedZoneConfig={
        'Comment': 'Production hosted zone',
        'PrivateZone': False
    }
)
zone_id = response['HostedZone']['Id'].split('/')[-1]
print(f"Hosted zone ID: {zone_id}")
print(f"Name servers: {response['DelegationSet']['NameServers']}")

In [ ]:
# Create an A record (simple routing)
r53.change_resource_record_sets(
    HostedZoneId=zone_id,
    ChangeBatch={
        'Changes': [
            {
                'Action': 'CREATE',
                'ResourceRecordSet': {
                    'Name': 'api.example.com',
                    'Type': 'A',
                    'TTL': 300,
                    'ResourceRecords': [
                        {'Value': '1.2.3.4'}
                    ]
                }
            }
        ]
    }
)
print("A record created")

In [ ]:
# Create an Alias record pointing to an ALB (zone apex — example.com itself)
r53.change_resource_record_sets(
    HostedZoneId=zone_id,
    ChangeBatch={
        'Changes': [
            {
                'Action': 'CREATE',
                'ResourceRecordSet': {
                    'Name': 'example.com',   # zone apex — CNAME not allowed here
                    'Type': 'A',
                    'AliasTarget': {
                        'HostedZoneId': 'Z35SXDOTRQ7X7K',  # ALB hosted zone ID (per region)
                        'DNSName': 'my-alb-1234.us-east-1.elb.amazonaws.com',
                        'EvaluateTargetHealth': True
                    }
                }
            }
        ]
    }
)
print("Alias record created for zone apex → ALB")

In [ ]:
# Weighted routing — 90% to v1, 10% to v2 (canary deployment)
for version, ip, weight in [('v1', '10.0.1.10', 90), ('v2', '10.0.2.10', 10)]:
    r53.change_resource_record_sets(
        HostedZoneId=zone_id,
        ChangeBatch={
            'Changes': [{
                'Action': 'CREATE',
                'ResourceRecordSet': {
                    'Name': 'app.example.com',
                    'Type': 'A',
                    'SetIdentifier': version,
                    'Weight': weight,
                    'TTL': 60,
                    'ResourceRecords': [{'Value': ip}]
                }
            }]
        }
    )
    print(f"{version}: {weight}% → {ip}")

In [ ]:
# Create a health check for an HTTP endpoint
hc = r53.create_health_check(
    CallerReference=str(uuid.uuid4()),
    HealthCheckConfig={
        'IPAddress': '1.2.3.4',
        'Port': 443,
        'Type': 'HTTPS',
        'ResourcePath': '/health',
        'FullyQualifiedDomainName': 'api.example.com',
        'RequestInterval': 30,       # seconds between checks
        'FailureThreshold': 3,       # consecutive failures before unhealthy
        'EnableSNI': True
    }
)
hc_id = hc['HealthCheck']['Id']
print(f"Health check: {hc_id}")

In [ ]:
# Failover routing — primary in us-east-1, secondary in eu-west-1
for role, dns, zone in [
    ('PRIMARY',   'primary-alb.us-east-1.elb.amazonaws.com',  'Z35SXDOTRQ7X7K'),
    ('SECONDARY', 'secondary-alb.eu-west-1.elb.amazonaws.com', 'Z32O12XQLNTSW2')
]:
    record = {
        'Name': 'api.example.com',
        'Type': 'A',
        'SetIdentifier': role,
        'Failover': role,
        'AliasTarget': {
            'HostedZoneId': zone,
            'DNSName': dns,
            'EvaluateTargetHealth': True
        }
    }
    # Primary requires a health check
    if role == 'PRIMARY':
        record['HealthCheckId'] = hc_id

    r53.change_resource_record_sets(
        HostedZoneId=zone_id,
        ChangeBatch={'Changes': [{'Action': 'CREATE', 'ResourceRecordSet': record}]}
    )
    print(f"{role} failover record created")

## Summary

| Concept | Key Takeaway |
|---|---|
| Public hosted zone | DNS for internet-facing domains |
| Private hosted zone | DNS within VPCs only — needs enableDnsSupport + enableDnsHostnames |
| CNAME | Maps hostname to hostname; cannot be used at zone apex |
| Alias | Maps hostname to AWS resource; free queries; allowed at zone apex |
| Simple routing | Single resource; no health checks; client picks if multiple values |
| Weighted routing | Split traffic by weight; A/B testing; canary deployments |
| Latency routing | Route to lowest-latency AWS region; not geography-based |
| Failover routing | Active/passive DR; health check required on primary |
| Geolocation routing | Route by user's country/continent; compliance and localisation |
| Geoproximity routing | Distance + bias to shift traffic; requires Traffic Flow |
| IP-based routing | Route by client CIDR block |
| Multi-value routing | Up to 8 healthy records; health-check-aware; not a load balancer |
| Health checks — endpoint | HTTP/HTTPS/TCP probe from public internet |
| Health checks — private | CloudWatch alarm → Route 53 alarm health check |
| Resolver inbound | On-premises resolves AWS private hosted zone names |
| Resolver outbound | EC2 resolves on-premises DNS via forwarding rules |